# Real World MCP: Augmenting Model Context from Existing Data Sources

Today we will use **Google MCP Toolbox** to connect agents to two local databases running in Docker. We will build an LLM-based agent with **OpenRouter**, wire up tool calling, and drive everything with **YAML tool definitions** so the agent can answer questions in a chat-like way.

After you run the notebook, you will practice your own queries and extend the toolbox with new tools.

### Quickstart

Run these steps in your terminal from the **repository root** unless noted otherwise.

1. **Python environment** — Create a venv and install the project in editable mode:

    ```bash
    python3.12 -m venv .venv

    # macOS / Linux:
    source .venv/bin/activate
    # Windows cmd:
    .venv\Scripts\activate.bat
    # Windows PowerShell:
    .venv\Scripts\Activate.ps1

    pip install -U pip
    pip install -e .
    ```

2. **Environment file** — Copy [`.env.example`](../.env.example) to `.env` in the repo root.

3. **OpenRouter** — Put your OpenRouter API key in the `OPENROUTER_API_KEY` section of `.env` (this is the main secret you must supply).

4. **Other Configurations** — Feel free to change other sections of `.env` such as the model you will be using.

4. **Databases** — Start Postgres, Mongo, and the seed job:

    ```bash
    docker compose up --build
    ```


**Troubleshooting:** See the main [README](../README.md) if anything fails.  

---

# I. Tutorial

### Dataset and Databases
**Our dataset is mirrored from [**126 Years of Historical Olympic Data**](https://www.kaggle.com/datasets/muhammadehsan02/126-years-of-historical-olympic-dataset?select=Olympic_Athlete_Event_Details.csv) on Kaggle. Running the docker-compose file creates and populates two databases:**

#### A Postgres (SQL) database with 4 tables:  
- **olympic_games** provides a summary of each Olympic Games edition, including key details such as the year, host city, and country.
- **athletes** is a slim table capturing athlete name, gender, and country of birth (as well as the primary key athlete_id referenced in other tables).
- **countries** provides a mapping of National Olympic Committee (NOC) codes to country names.
- **athlete_events** contains detailed results of Olympic events for each athlete from 1896 to 2022. Each row represents one athlete's participation in a single event; each event can have multiple participants, and each athlete can have participated in multiple events.

#### A MongoDB document database with 2 collections:
- **olympic_athlete_biography** provides detailed biographical information on each Olympic athlete represented in the dataset.
- **olympic_event_results** contains detailed results of each Olympic event represented in the dataset.

## 1. MCP Toolbox Configuration

**The docker-compose file also sets up our MCP server:**
- The Toolbox service wraps our databases in a local, unsecured MCP server.
- The tools our agent will use are configured at runtime via the YAML files in the `/toolbox/config/tools` folder.
- At the end of this notebook you will use these same files to define your own custom tools.

Advanced concepts such as security/auth and additional database configuration guidance can be found in the official [MCP Toolbox for Databases](https://mcp-toolbox.dev/documentation/introduction/) documentation.

### 1.1 **Sources**

Our first YAML file ([**01_sources.yaml**](../toolbox/config/tools/01_sources.yaml)) defines our **sources** - these are the named database connections against which we will run our queries. As we define our tools, we will point to specific sources using `source: <name>`. Below is the source for our Postgres database (notice that we are pulling most of these values from our environment variables).

```yaml
kind: source
name: olympics-postgres
type: postgres
host: ${POSTGRES_HOST}
port: ${POSTGRES_PORT}
database: ${POSTGRES_DB}
user: ${POSTGRES_USER}
password: ${POSTGRES_PASSWORD}
```

### 1.2 **Tools**

The other YAML documents in our toolbox directory define our **tools**, which our agent will use to interact with our databases. MCP Toolbox automatically merges every `*.yaml` under `toolbox/config/tools/`, so we can subdivide our tools into logical groups.

Below you can see `pg_search_countries` from [`02_postgres_tools.yaml`](../toolbox/config/tools/02_postgres_tools.yaml). This file defines all of the tools for our Postgres database, and this particular tool is designed to look up countries based on their NOC code.

We can see that this tool is identified by a unique `name`, runs a `statement` (in this case a SQL query) on the linked `source`, and identifies the parameters the model will fill in (mapped to `$1`, `$2`, ... in order).

```yaml
kind: tool
name: pg_search_countries
type: postgres-sql
source: olympics-postgres
description: >
  Search countries by name fragment and return NOC codes (National Olympic Committee).
  Use before medal tools when the user says e.g. "United States" instead of "USA".
parameters:
  - name: name_fragment
    type: string
    description: Substring to match against country name (case-insensitive).
statement: |
  SELECT noc, country
  FROM countries
  WHERE country ILIKE '%' || $1 || '%'
  ORDER BY country ASC
  LIMIT 50
annotations:
  readOnlyHint: true
  destructiveHint: false
```

### 1.5 **Toolsets**

Toolsets (defined for our MCP server in [**04_toolsets.yaml**](toolbox/config/tools/04_toolsets.yaml)) are named bundles of tools, which the MCP server can expose to agents as a group. 

In this case, we have declared three toolsets:
- **postgres_only** - only the Postgres SQL tools (pg_*).
- **mongo_only** — only the Mongo tools (mongo_*).
- **combined** — all of those Postgres and Mongo tools in one list.

MCP clients (like the one we are about to build) can use these toolsets to fetch subsets of tools that are relevant to specific tasks. This serves as an important means to control which agents have access to which tools, and under what circumstances. It also means that we do not have to include all of our tool definitions in every API call to the LLM powering our agent (which for a larger corpus of tools can greatly increase token usage and the cost of running an agent).

### 1.4 **Prompts**

MCP Toolbox also supports storing **prompts** in our yaml files. This is beyond the scope of this tutorial, but a useful way to organize the various system prompts you want your agents to use for specific tasks. We have a few examples [here](../toolbox/config/tools/05_prompts.yaml) to demonstrate the concept.

---

## 2. Environment and Settings

Before we build our agent, let's import our Python libraries and load our settings.

#### Key libraries

| Package | Role in this tutorial |
| :--- | :--- |
| **`python-dotenv`** (`dotenv`) | Loads `.env` so API keys and URLs (OpenRouter, Toolbox) are available without hard-coding secrets. |
| **`openai`** | Official OpenAI SDK in **OpenAI-compatible** mode: OpenRouter uses the same request/response shapes, so we use it for chat completions and tool/function calling with OpenRouter credentials. |
| **`toolbox-core`** | Google **MCP Toolbox** Python client: connects to the local Toolbox MCP server, discovers tools from our YAML files, and invokes them as `ToolboxSyncTool` over the MCP protocol. |

**Docs:** [OpenRouter quickstart](https://openrouter.ai/docs/quickstart) (keys and OpenAI-compatible base URL); [`toolbox-core` on PyPI](https://pypi.org/project/toolbox-core/); [Python SDK core guide](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/) ([sync client](https://googleapis.github.io/genai-toolbox/sdks/python-sdk/core/#synchronous-usage)).


In [ ]:
import os
import json
import random
from typing import Any
from pathlib import Path
from dataclasses import dataclass
from collections.abc import Sequence

# OpenAI client (used to call OpenRouter for inference)
from openai import OpenAI
# Load environment variables
from dotenv import load_dotenv

# MCP Toolbox client
from toolbox_core import ToolboxSyncClient
from toolbox_core.protocol import Protocol
from toolbox_core.sync_tool import ToolboxSyncTool

# Maximum number of tool calls per user turn allowed for the model
AGENT_MAX_TOOL_ROUNDS = 16

@dataclass(frozen=True)
class Settings:
    """URLs and keys from the environment; ``agent_max_tool_rounds`` is fixed here."""

    openrouter_api_key: str
    openrouter_base_url: str
    openrouter_model: str
    toolbox_base_url: str
    agent_max_tool_rounds: int

    @staticmethod
    def from_env() -> "Settings":
        """Load settings from the environment."""

        key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
        base = (
            os.environ.get("OPENROUTER_BASE_URL") or "https://openrouter.ai/api/v1"
        ).strip()
        model = (os.environ.get("OPENROUTER_MODEL") or "openai/gpt-4o-mini").strip()
        toolbox = (
            (os.environ.get("TOOLBOX_BASE_URL") or "http://127.0.0.1:5050")
            .strip()
            .rstrip("/")
        )
        return Settings(
            openrouter_api_key=key,
            openrouter_base_url=base,
            openrouter_model=model,
            toolbox_base_url=toolbox,
            agent_max_tool_rounds=AGENT_MAX_TOOL_ROUNDS,
        )

def repo_root() -> Path:
    """Find the repo root folder by walking up the directory tree and looking for pyproject.toml."""

    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError("Run from inside the repo clone (need pyproject.toml).")

# Load settings from the .env file
load_dotenv(repo_root() / ".env")
settings = Settings.from_env()


---

## 3. Tool-calling loop

Now that we understand the basic building blocks of our MCP server, let's go step-by-step and build a simple agent using a **tool-calling loop**.

### 3.1 JSON Schema type names and Toolbox-to-OpenAI helper function

In order to utilize our tool definitions from the YAML files used by MCP Toolbox, we need a helper to convert them to the static JSON format expected in an LLM call. In the cell below, we first create a mapping between the data types used by each format, and then we implement the mapper function.



In [ ]:
# ``json`` parses each tool's argument string from the model into a Python dict.
# ``Sequence`` / ``Any`` keep type hints readable for ``ToolboxSyncTool`` lists and message dicts.

# Map to convert Toolbox types to JSON Schema types
_JSON_TYPES = {
    "string": "string",
    "integer": "integer",
    "float": "number",
    "boolean": "boolean",
    "array": "array",
    "object": "object",
}

def toolbox_tools_to_openai(tools: Sequence[ToolboxSyncTool]) -> list[dict[str, Any]]:
    """Map MCP Toolbox formatted tools to OpenAI chat completion tools format."""

    out: list[dict[str, Any]] = []
    for t in tools:
        # Gather the properties and required parameters for the tool
        props: dict[str, Any] = {}
        required: list[str] = []
        for p in t._params:
            props[p.name] = {
                "type": _JSON_TYPES.get(p.type, "string"),
                "description": p.description,
            }
            if p.required:
                required.append(p.name)
        # Create the OpenAI 'function' object
        out.append(
            {
                "type": "function",
                "function": {
                    "name": t._name,
                    "description": (t._description or "").strip(),
                    "parameters": {
                        "type": "object",
                        "properties": props,
                        "required": required,
                        "additionalProperties": False,
                    },
                },
            }
        )
    return out

### 3.2 Look up tools by function name

The LLM powering our agent does not run our Python tools directly; it will return the strings **function.name** and **function.arguments** for each tool it wants to use. Our code must run the tool, and feed the result back to the LLM.

This means that our next step is to build a helper function that builds a dictionary mapping each **function.name** string to the corresponding Toolbox tool object that our code will run.


In [ ]:
def tool_by_name(tools: Sequence[ToolboxSyncTool]) -> dict[str, ToolboxSyncTool]:
    """Map tools by name."""
    return {t._name: t for t in tools}


### 3.3 OpenRouter via the OpenAI SDK

Next we create a function to instantiate an OpenAI client. Note that we are pointing base_url at OpenRouter, which means we are using the **OpenAI format**, but route calls through **OpenRouter's endpoint** to access our LLM.

In [ ]:
def openrouter_client(settings: Settings) -> OpenAI:
    """Instantiate an OpenRouter inference client."""

    return OpenAI(
        # OpenAI's chat API is supported by OpenRouter.
        api_key=settings.openrouter_api_key,
        base_url=settings.openrouter_base_url,
        # Add headers to identify your application, if desired.
        default_headers={
            "HTTP-Referer": "https://github.com/Rotational-io/mcp-tutorial",
            "X-Title": "Rotational.io MCP Tutorial",
        },
    )


### 3.4 Build the first `messages` list

Each call to the chat API sends the whole conversation history as a list of messages. We build the first message with the usual two roles: system (behavior and constraints) and user (the actual question).

Later, when the model asks for tools, we will append an assistant turn (including the tool call ids) and one tool turn per result.

In [ ]:
def initial_messages(
    system_prompt: str,
    user_prompt: str,
    cap: int,
) -> list[dict[str, Any]]:
    """Build the first messages list."""

    # Keep the model's plan aligned with how many tool-bearing turns the client allows.
    sys_text = (
        system_prompt.rstrip()
        + f"\n\nAt most {cap} model turns may include tool calls."
    )
    # Chat Completions expects a time-ordered list; we will append assistant + tool rows as they are generated.
    return [
        {"role": "system", "content": sys_text},
        {"role": "user", "content": user_prompt},
    ]


### 3.5 Ask the model once

Next we wrap a single `chat.completions.create` call in a small helper. Each time we call it we send the full messages transcript built so far, and the same tools list (the Toolbox catalog we mapped to OpenAI’s shape). The model can answer in plain text, request tool calls, or both depending on the reply.

We set `tool_choice="auto"` so the API decides whether to invoke tools; we are not forcing a tool on every turn.


In [ ]:
def call_model(
    oai: OpenAI,
    settings: Settings,
    messages: list[dict[str, Any]],
    oai_tools: list[dict[str, Any]],
):
    """Perform a single chat completion round."""

    return oai.chat.completions.create(
        model=settings.openrouter_model,
        messages=messages,  # full transcript so far (system, user, prior tools, ...)
        tools=oai_tools,  # same tool catalog every round
        tool_choice="auto",  # model may answer with text only, call tools, or mix
    )


### 3.6 Record tool calls and append tool results

When the model’s reply includes tool_calls, the chat API expects you to extend the transcript in a strict order before you call the model again:

1. Append an assistant message that replays those calls: same `id`, `function.name`, and `function.arguments` (still a JSON string) for each one. That row is the record of what the model asked for.  

2. Append one tool message per call. Each tool row must use the matching `tool_call_id` (same value as the `id` from step 1) and put the tool output in content as a string—here that string is whatever Toolbox returns from running the tool.

If either step is missing or the ids do not line up, the next completion request is invalid. The two helpers in the next cell adhere to that pattern: first mirror the assistant turn, then run each tool and append the results.


In [ ]:
def append_assistant_with_tool_calls(messages: list[dict[str, Any]], msg) -> None:
    """Append the assistant generated messages and tool calls to the messages list."""

    messages.append(
        {
            "role": "assistant",
            "content": msg.content,  # content will be empty when the model only wants tools
            "tool_calls": [
                {
                    "id": tc.id,  # tool call id is linked to the tool call results later
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,  # JSON object as a string
                    },
                }
                for tc in msg.tool_calls
            ],
        }
    )


def call_tool_and_append_tool_outputs(
    messages: list[dict[str, Any]],
    msg,
    by_name: dict[str, ToolboxSyncTool],
) -> None:
    """Call Toolbox tools and append the results to the messages list."""

    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        # Call the Toolbox tool by name and append the result to the messages list.
        payload = by_name[tc.function.name](**args)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": payload,
            }
        )


### 3.7 Put it together: `chat_with_tools`

Now we are ready to put all of this together in a single `chat_with_tools` function. We open one MCP session to Toolbox for this user turn (so every tool call shares the same connection) then load the combined toolset (which we defined in `04_toolsets.yaml`). From that we use our helper functions to build the OpenAI-style tools list for the model and the name-to-tool map our code uses to execute each tool call selected by the model.

After that we have a simple loop. We call `call_model` with whatever messages we have so far. If the model comes back with `tool_calls` we append the assistant row, run the tools, append the tool results, and ask again with another `call_model` call. If it answers with plain text and no tool calls, we are done; that string is the reply. If we hit the round cap before we receive our reply, we exit the loop and return a “round limit” message instead.

Note that within this function we have also implemented a `verbose` flag that will allow us to toggle between printing only the agent's answer and seeing the whole list of tool calls and "conversation" the agent has while interacting with our databases.

In [ ]:
def chat_with_tools(
    settings: Settings,
    *,
    user_prompt: str,
    system_prompt: str,
    max_rounds: int | None = None,
    verbose: bool = False,
) -> str:
    """Perform several rounds of tool use and model inference."""

    # Cap the number of tool calls per user turn
    cap = max_rounds if max_rounds is not None else settings.agent_max_tool_rounds

    # Initialize the OpenAI client and the messages list
    oai = openrouter_client(settings)
    messages = initial_messages(system_prompt, user_prompt, cap)

    # Limit the length of tool output
    _tool_out_limit = 4000

    # Keep the client open for the whole user turn so every tool call shares one MCP session.
    with ToolboxSyncClient(
        settings.toolbox_base_url,
        protocol=Protocol.MCP_LATEST,
    ) as tb:
        # Load the combined toolset and process tools for later use
        loaded = tb.load_toolset("combined")  # name from toolbox/config/tools/04_toolsets.yaml
        by_name = tool_by_name(loaded)
        oai_tools = toolbox_tools_to_openai(loaded)

        # Perform each round of tool use and model inference; ends when there are no more tool calls or the round limit is reached
        for round_idx in range(cap):
            # Call the model with the current messages and tools
            resp = call_model(oai, settings, messages, oai_tools)

            # Get the model's response for this round (if any)
            msg = resp.choices[0].message
            if verbose:
                print(f"\n--- model turn {round_idx + 1} ---", flush=True)
                if msg.content and str(msg.content).strip():
                    print("assistant (text):", str(msg.content).strip(), flush=True)
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        args = json.loads(tc.function.arguments or "{}")
                        print(f"tool_call id={tc.id} name={tc.function.name}", flush=True)
                        print(json.dumps(args, indent=2), flush=True)

            # If the model wants to use tools, record what it asked for, run the tools, and loop back to the model with the results.
            if msg.tool_calls:
                append_assistant_with_tool_calls(messages, msg)
                call_tool_and_append_tool_outputs(messages, msg, by_name)
                if verbose:
                    for row in messages[-len(msg.tool_calls) :]:
                        body = row["content"]
                        if len(body) > _tool_out_limit:
                            body = body[:_tool_out_limit] + "\n… [truncated]"
                        print(f"tool_result id={row['tool_call_id']}", flush=True)
                        print(body, flush=True)
                continue

            # No tool calls: treat the assistant's text as the final answer for this user message.
            if verbose:
                print("\n--- final assistant reply ---", flush=True)
                print((msg.content or "").strip(), flush=True)
            return (msg.content or "").strip()

    # If we exit the loop without returning a reply, we have hit the round limit.
    if verbose:
        print("\n--- round limit (no text-only finish) ---", flush=True)
    return "Round limit reached without a final text reply."


---

## 4. Examples
Now let's put our agent to work and use it to ask some questions about historical Olympic data:

### Example A — structured data then bios

Here we are going to ask our agent a question that allows it to make some choices. After giving it the persona of a "tutorial assistant with Olympic data tools" in the system prompt, we give it a range of years from which to select a specific Olympic Games, and a list of countries to choose from. We then ask it to return information from the biographies of some of the athletes from that country who medaled during those games.

This is an interesting query because it not only allows the agent to make some unique choices, but also requires it to combine information that is stored separately in the Postgres and Mongo databases. Try running the cell as-is, and review the results. Then uncomment the line with the `verbose` flag, and run it again to follow the tool calls and the "conversation" the agent has while interacting with our databases. Does its workflow match what you would expect?

In [ ]:
TUTORIAL_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Keep tool use reasonable in depth (a few biography lookups per answer is enough). "
    "Prefer concise answers with citations to ids returned by tools."
)

PROMPT_A = (
    "Choose a Summer Olympic Games held between 1992 and 2020, and choose one of these "
    "countries to focus on: United States, Australia, Japan, France, or Kenya. Who won "
    "medals for that country at those Games (any sport)? Use the data tools, mention a "
    "few medalists with a line or two from their biographies when you can, and cite "
    "athlete ids from the tools."
)

answer_a = chat_with_tools(
    settings,
    user_prompt=PROMPT_A,
    system_prompt=TUTORIAL_SYSTEM_PROMPT,
    #verbose=True
)
print(answer_a)

### Example B — biography search then cross-check

Here we flip the story from Example A. We still use the same tutorial assistant setup, but the user prompt starts from a sport theme in plain language (this notebook picks one phrase at random from a small list). We expect this to nudge the model toward a text-oriented search over the athlete biographies with mongo_* tools first, then follow-up calls to retrieve structured information Postgres. The answer should compare what the bio text emphasizes with what the Postgres database actually records.

Once again, start by running the cell below with the `verbose` flag turned off; then run it again with the flag un-commented. Do the final results vary between runs? Does the agent behavior match what you expect? How is it different from the previous question?

In [ ]:
TUTORIAL_SYSTEM_PROMPT = (
    "You are a tutorial assistant with Olympic data tools (relational + document). "
    "Use tools for facts; do not invent athlete, edition, or country codes. "
    "Call tools yourself—do not ask the user to run queries. "
    "Keep tool use reasonable in depth (a few biography lookups per answer is enough). "
    "Prefer concise answers with citations to ids returned by tools."
)

NOTEBOOK_B_SPORTS = (
    "decathlon",
    "pole vault",
    "100 metres freestyle",
    "balance beam",
    "badminton",
)


def notebook_prompt_b() -> tuple[str, str]:
    sport = random.choice(NOTEBOOK_B_SPORTS)
    user = (
        f"Find athlete biography material tied to Olympic sport or event, using the theme "
        f"{sport!r} in a text-oriented search over the biography data. "
        f"Then cross-check that person against structured Olympic tables (starts, medals, editions) "
        f"and summarize: what the bio emphasizes versus what the relational data supports. "
        f"Cite ids where helpful."
    )
    return user, sport


prompt_b, sport_b = notebook_prompt_b()
print("Sport theme:", sport_b)

answer_b = chat_with_tools(
    settings,
    user_prompt=prompt_b,
    system_prompt=TUTORIAL_SYSTEM_PROMPT,
    #verbose=True
)
print(answer_b)

---

# II. Individual Practice

Now that we have built our tool-calling loop and executed some example calls, let's practice using some of the other tools defined in our [toolbox](../toolbox/config/tools). After that we will implement new tools on our own, and use them to execute additional tool calls.